In [1]:
import os, glob, subprocess, itertools
import numpy as np, pandas as pd

subprocess.run(["pip","-q","install","freesasa","biopython"], check=True)
import freesasa
freesasa.setVerbosity(freesasa.silent)

from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

PROJ = "/content/drive/MyDrive/il2_mimetic"
assert os.path.isfile(f"{PROJ}/stage1_motif/params.json"), "wrong Drive account?"
OUT4 = f"{PROJ}/stage4_fold"
PRED = f"{PROJ}/stage5_complex/preds"
BIO  = f"{PROJ}/analysis_biophys"; os.makedirs(BIO, exist_ok=True)
TMP  = "/content/_bio"; os.makedirs(TMP, exist_ok=True)

NONPOLAR = set("CS")          # carbon, sulfur
def element(line):
    e = line[76:78].strip()
    return e if e else line[12:16].strip().lstrip("0123456789")[0]

def atom_lines(path, chains=None):
    out = []
    for l in open(path):
        if l.startswith("ATOM") and (chains is None or l[21] in chains):
            out.append(l)
    return out

def sasa_of(lines, tag):
    """Total, polar and nonpolar SASA for a set of ATOM lines."""
    p = f"{TMP}/{tag}.pdb"
    with open(p, "w") as fh:
        fh.writelines(lines); fh.write("END\n")
    st = freesasa.Structure(p)
    res = freesasa.calc(st)
    tot = npol = 0.0
    for i in range(st.nAtoms()):
        a = res.atomArea(i)
        tot += a
        if element(lines[i]) in NONPOLAR:
            npol += a
    return tot, npol, tot - npol

def interface(path, c1, c2, tag):
    """Buried surface area between two chains, and its nonpolar fraction."""
    l1, l2 = atom_lines(path, {c1}), atom_lines(path, {c2})
    if not l1 or not l2: return None
    t1, n1, _ = sasa_of(l1, f"{tag}_1")
    t2, n2, _ = sasa_of(l2, f"{tag}_2")
    tc, nc, _ = sasa_of(l1 + l2, f"{tag}_c")
    bsa  = (t1 + t2 - tc) / 2.0          # per side
    bnp  = (n1 + n2 - nc) / 2.0
    return dict(bsa=bsa, bsa_nonpolar=bnp,
                frac_nonpolar=bnp / bsa if bsa > 1 else np.nan)

def pred_pdb(job):
    f = glob.glob(f"{PRED}/boltz_results_{job}/predictions/{job}/*.pdb")
    return f[0] if f else None

jobs = sorted(os.path.basename(f)[:-6]
              for f in glob.glob(f"{PROJ}/stage5_complex/inputs/*.fasta"))

rows = []
for job in jobs:
    p = pred_pdb(job)
    if not p: continue
    design = job.rsplit("_", 1)[0]
    if job.endswith("_ternary"):
        for lbl, c2 in [("beta","B"), ("gc","C")]:
            r = interface(p, "A", c2, f"{job}_{lbl}")
            if r: rows.append(dict(design=design, partner=lbl, **r))
        r = interface(p, "B", "C", f"{job}_bgc")
        if r: rows.append(dict(design=design, partner="beta_gc", **r))
    else:
        r = interface(p, "A", "B", f"{job}_alpha")
        if r: rows.append(dict(design=design, partner="alpha", **r))

bsa = pd.DataFrame(rows)
piv = bsa.pivot(index="design", columns="partner",
                values=["bsa","frac_nonpolar"])
piv.columns = [f"{a}_{b}" for a, b in piv.columns]
order = ["ctrl_IL2","ctrl_Neo","ctrl_scram"] + \
        sorted(d for d in piv.index if not d.startswith("ctrl_"))
piv = piv.reindex([d for d in order if d in piv.index])

pd.set_option("display.width", 220)
print(piv.round(3).to_string())
piv.round(4).to_csv(f"{BIO}/interface_bsa.tsv", sep="\t")

Mounted at /content/drive
                              bsa_alpha  bsa_beta  bsa_beta_gc   bsa_gc  frac_nonpolar_alpha  frac_nonpolar_beta  frac_nonpolar_beta_gc  frac_nonpolar_gc
design                                                                                                                                                   
ctrl_IL2                       1138.100   664.858      909.745  455.072                0.501               0.583                  0.489             0.630
ctrl_Neo                       1028.057   741.884      998.467  489.367                0.590               0.562                  0.484             0.707
ctrl_scram                     1037.599   773.241      917.267  524.781                0.581               0.600                  0.454             0.669
pad0_len100_13_fixContacts_3   1221.022   671.809      972.086  430.289                0.585               0.591                  0.475             0.596
pad0_len100_15_fixAll_4        1443.851   495.028 

In [2]:
from scipy.spatial import cKDTree

POLAR = set("NO")

def buns(lines, tag, burial_cut=1.0, hb_cut=3.5):
    """Buried polar atoms with no polar neighbour within hb_cut Å."""
    p = f"{TMP}/{tag}.pdb"
    with open(p,"w") as fh: fh.writelines(lines); fh.write("END\n")
    st  = freesasa.Structure(p); res = freesasa.calc(st)

    xyz  = np.array([[float(l[30+8*i:38+8*i]) for i in range(3)] for l in lines])
    elem = [element(l) for l in lines]
    resid= [(l[21], l[22:27]) for l in lines]
    area = np.array([res.atomArea(i) for i in range(st.nAtoms())])

    polar_idx = [i for i,e in enumerate(elem) if e in POLAR]
    tree = cKDTree(xyz[polar_idx])
    pairs = tree.query_ball_point(xyz[polar_idx], r=hb_cut)

    n_polar = len(polar_idx)
    n_buried = n_unsat = 0
    for k, i in enumerate(polar_idx):
        if area[i] > burial_cut: continue          # not buried
        n_buried += 1
        partners = [polar_idx[j] for j in pairs[k]
                    if polar_idx[j] != i and resid[polar_idx[j]] != resid[i]]
        if not partners: n_unsat += 1
    return dict(n_polar=n_polar, n_buried_polar=n_buried,
                n_buns=n_unsat,
                buns_per_100res=100*n_unsat/len({r for r in resid}))

rows = []
for job in [j for j in jobs if j.endswith("_ternary")]:
    p = pred_pdb(job)
    if not p: continue
    design = job.rsplit("_",1)[0]
    chainA = atom_lines(p, {"A"})
    rows.append(dict(design=design, **buns(chainA, f"{job}_mono")))

bn = pd.DataFrame(rows).set_index("design")
bn = bn.reindex([d for d in order if d in bn.index])
print(bn.round(2).to_string())
bn.round(3).to_csv(f"{BIO}/buried_unsat_polars.tsv", sep="\t")

                              n_polar  n_buried_polar  n_buns  buns_per_100res
design                                                                        
ctrl_IL2                          380             191       3             2.26
ctrl_Neo                          304             162       0             0.00
ctrl_scram                        316             148       3             2.86
pad0_len100_13_fixContacts_3      297             164       2             2.04
pad0_len100_15_fixAll_4           316             167       2             1.90
pad0_len100_22_fixAll_0           323             178       1             0.93
pad0_len100_2_fixContacts_3       349             189       3             2.61
pad0_len100_37_fixContacts_2      332             184       1             0.90
pad2_len100_7_fixAll_3            256             139       0             0.00


In [3]:
BACKBONE = {"N", "O", "OXT"}

def buns_sc(lines, tag, burial_cut=1.0, hb_cut=3.2):
    """Buried SIDECHAIN polar atoms with no polar partner in another residue."""
    p = f"{TMP}/{tag}.pdb"
    with open(p,"w") as fh: fh.writelines(lines); fh.write("END\n")
    st, res = freesasa.Structure(p), None
    res = freesasa.calc(st)

    xyz   = np.array([[float(l[30+8*i:38+8*i]) for i in range(3)] for l in lines])
    elem  = [element(l) for l in lines]
    name  = [l[12:16].strip() for l in lines]
    resid = [(l[21], l[22:27]) for l in lines]
    area  = np.array([res.atomArea(i) for i in range(st.nAtoms())])

    all_polar = [i for i,e in enumerate(elem) if e in POLAR]
    tree = cKDTree(xyz[all_polar])

    sc_polar = [i for i in all_polar if name[i] not in BACKBONE]
    n_buried = n_unsat = 0
    unsat_res = []
    for i in sc_polar:
        if area[i] > burial_cut: continue
        n_buried += 1
        near = tree.query_ball_point(xyz[i], r=hb_cut)
        partners = [all_polar[j] for j in near
                    if all_polar[j] != i and resid[all_polar[j]] != resid[i]]
        if not partners:
            n_unsat += 1
            unsat_res.append(f"{lines[i][17:20].strip()}{resid[i][1].strip()}:{name[i]}")
    nres = len({r for r in resid})
    return dict(n_sc_polar=len(sc_polar), n_buried_sc=n_buried,
                buns_sc=n_unsat, buns_sc_per_100res=100*n_unsat/nres,
                where=";".join(unsat_res[:6]))

rows = []
for job in [j for j in jobs if j.endswith("_ternary")]:
    p = pred_pdb(job)
    if not p: continue
    rows.append(dict(design=job.rsplit("_",1)[0],
                     **buns_sc(atom_lines(p, {"A"}), f"{job}_sc")))

bs = pd.DataFrame(rows).set_index("design").reindex(
        [d for d in order if d in pd.DataFrame(rows).set_index("design").index])
print(bs.to_string())
bs.to_csv(f"{BIO}/buried_unsat_sidechain.tsv", sep="\t")

                              n_sc_polar  n_buried_sc  buns_sc  buns_sc_per_100res                                             where
design                                                                                                                              
ctrl_IL2                             114            9        4            3.007519           ASN33:ND2;GLU60:OE1;SER75:OG;TRP121:NE1
ctrl_Neo                              96            2        0            0.000000                                                  
ctrl_scram                           106           13        3            2.857143                       ASN8:ND2;ASN64:ND2;TYR97:OH
pad0_len100_13_fixContacts_3         101            5        2            2.040816                                 GLN6:NE2;ARG24:NE
pad0_len100_15_fixAll_4              106            4        3            2.857143                     GLN61:NE2;GLU69:OE1;TRP89:NE1
pad0_len100_22_fixAll_0              109            5        2       

In [5]:
import subprocess
subprocess.run(["pip","-q","install","py3Dmol"], check=True)

p = pred_pdb("pad0_len100_2_fixContacts_3_ternary")
bad = ["12","22","60","92","96"]          # from the `where` column

import py3Dmol
v = py3Dmol.view(width=850, height=560)
v.addModel("".join(atom_lines(p, {"A"})), "pdb")
v.setStyle({"cartoon":{"color":"#d9d9d9"}})
v.addStyle({"resi": bad}, {"stick":{"colorscheme":"magentaCarbon","radius":0.22}})
v.addStyle({"resi": bad}, {"sphere":{"opacity":0.25,"color":"#CC79A7"}})
v.zoomTo({"resi": bad}); v.show()
print("magenta = buried unsatisfied sidechain polars")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

magenta = buried unsatisfied sidechain polars


In [6]:
subprocess.run(["pip","-q","install","freesasa","biopython","py3Dmol"], check=True)

CompletedProcess(args=['pip', '-q', 'install', 'freesasa', 'biopython', 'py3Dmol'], returncode=0)

In [7]:
lines = atom_lines(p, {"A"})
with open(f"{TMP}/chk.pdb","w") as fh: fh.writelines(lines); fh.write("END\n")
st = freesasa.Structure(f"{TMP}/chk.pdb"); res = freesasa.calc(st)
xyz = np.array([[float(l[30+8*i:38+8*i]) for i in range(3)] for l in lines])
elem = [element(l) for l in lines]
polar = [i for i,e in enumerate(elem) if e in POLAR]

for i, l in enumerate(lines):
    rn = l[22:27].strip()
    if rn in bad and l[12:16].strip() not in BACKBONE and elem[i] in POLAR:
        d = [np.linalg.norm(xyz[i]-xyz[j]) for j in polar
             if j != i and (lines[j][21], lines[j][22:27]) != (l[21], l[22:27])]
        print(f"{l[17:20]}{rn:>4s} {l[12:16].strip():<4s} "
              f"SASA {res.atomArea(i):5.2f} Å²   nearest polar {min(d):.2f} Å")

TYR  12 OH   SASA  0.79 Å²   nearest polar 4.37 Å
ARG  22 NE   SASA  1.93 Å²   nearest polar 5.52 Å
ARG  22 NH1  SASA  0.24 Å²   nearest polar 4.84 Å
ARG  22 NH2  SASA  9.12 Å²   nearest polar 4.28 Å
ASN  60 OD1  SASA  0.00 Å²   nearest polar 3.94 Å
ASN  60 ND2  SASA 18.77 Å²   nearest polar 3.40 Å
THR  92 OG1  SASA  0.00 Å²   nearest polar 3.26 Å
HIS  96 ND1  SASA  0.91 Å²   nearest polar 3.39 Å
HIS  96 NE2  SASA  3.60 Å²   nearest polar 3.30 Å


In [8]:
for cut in (3.2, 3.5):
    rows = []
    for job in [j for j in jobs if j.endswith("_ternary")]:
        pp = pred_pdb(job)
        if not pp: continue
        rows.append(dict(design=job.rsplit("_",1)[0],
                         **buns_sc(atom_lines(pp,{"A"}), f"{job}_{cut}", hb_cut=cut)))
    t = pd.DataFrame(rows).set_index("design")[["buns_sc"]]
    t.columns = [f"buns@{cut}"]
    print(t.reindex([d for d in order if d in t.index]).to_string(), "\n")

                              buns@3.2
design                                
ctrl_IL2                             4
ctrl_Neo                             0
ctrl_scram                           3
pad0_len100_13_fixContacts_3         2
pad0_len100_15_fixAll_4              3
pad0_len100_22_fixAll_0              2
pad0_len100_2_fixContacts_3          5
pad0_len100_37_fixContacts_2         1
pad2_len100_7_fixAll_3               0 

                              buns@3.5
design                                
ctrl_IL2                             3
ctrl_Neo                             0
ctrl_scram                           3
pad0_len100_13_fixContacts_3         2
pad0_len100_15_fixAll_4              2
pad0_len100_22_fixAll_0              1
pad0_len100_2_fixContacts_3          3
pad0_len100_37_fixContacts_2         1
pad2_len100_7_fixAll_3               0 



In [9]:
#### Checking out the sister sequence that fo selected side chains removes sticky residues that destabilize protein ####
import pandas as pd
seqs = pd.read_csv(f"{PROJ}/stage3_seqs/sequences.tsv", sep="\t")
parent = "pad0_len100_2_fixContacts"          # strip the trailing sequence index
sibs = seqs[seqs.seq_id.str.startswith(parent)]
print(f"{len(sibs)} sequences from this backbone\n")
for _, r in sibs.iterrows():
    print(f"{r.seq_id:34s} pos22={r.sequence[21]}  {r.sequence[16:28]}")

5 sequences from this backbone

pad0_len100_2_fixContacts_0        pos22=Y  LDLQMYKELIEK
pad0_len100_2_fixContacts_1        pos22=Y  LDLQMYKELVEQ
pad0_len100_2_fixContacts_2        pos22=L  LDLQMLKEKIEE
pad0_len100_2_fixContacts_3        pos22=R  LDLQMRKEKIKE
pad0_len100_2_fixContacts_4        pos22=Y  LDLQMYKKKVEE


Three siblings have tyrosine, one has leucine.
Leucine is fully nonpolar, so it cannot be an unsatisfied buried polar by construction. Tyrosine still has a hydroxyl and it was flagged.

In [10]:
# Stage 4 gate results for all five siblings
fm = pd.read_csv(f"{PROJ}/stage4_fold/fold_metrics.tsv", sep="\t")
sib_ids = sibs.seq_id.tolist()
cols = [c for c in fm.columns if c != "sequence"]
print(fm[fm.seq_id.isin(sib_ids)][cols].round(3).to_string(index=False))

# buns on the ESMFold models — same source for all five, so it is a fair comparison
rows = []
for sid in sib_ids:
    p = f"{PROJ}/stage4_fold/pdb/{sid}.pdb"
    if not os.path.exists(p):
        print("no ESMFold model:", sid); continue
    r = buns_sc(atom_lines(p), f"sib_{sid}", hb_cut=3.5)
    rows.append(dict(seq_id=sid, pos22=seqs.loc[seqs.seq_id==sid,"sequence"].iloc[0][21], **r))
print("\n", pd.DataFrame(rows).set_index("seq_id").to_string())

                     seq_id         arm     variant      backbone  length  mpnn_score  plddt  sc_rmsd  motif_rmsd  pass_plddt  pass_scrmsd  pass_motif  pass   q
pad0_len100_2_fixContacts_0 fixContacts pad0_len100 pad0_len100_2     115       1.213  91.46    0.702       0.471        True         True        True  True 3rd
pad0_len100_2_fixContacts_1 fixContacts pad0_len100 pad0_len100_2     115       1.199  88.97    0.913       0.435       False         True        True False 2nd
pad0_len100_2_fixContacts_2 fixContacts pad0_len100 pad0_len100_2     115       1.193  86.79    1.185       0.362       False         True        True False 2nd
pad0_len100_2_fixContacts_3 fixContacts pad0_len100 pad0_len100_2     115       1.224  92.01    0.552       0.317        True         True        True  True 3rd
pad0_len100_2_fixContacts_4 fixContacts pad0_len100 pad0_len100_2     115       1.189  91.82    1.744       0.329        True        False        True False 2nd

                             pos2

In [11]:
# build the two Stage 5 inputs for _0 by copying _3's and swapping chain A
import pandas as pd
IN = f"{PROJ}/stage5_complex/inputs"
NEW, OLD = "pad0_len100_2_fixContacts_0", "pad0_len100_2_fixContacts_3"

seqs = pd.read_csv(f"{PROJ}/stage3_seqs/sequences.tsv", sep="\t")
new_seq = seqs.loc[seqs.seq_id == NEW, "sequence"].iloc[0]

for panel in ("ternary", "alpha"):
    src = open(f"{IN}/{OLD}_{panel}.fasta").read().strip().split("\n")
    out, swap = [], False
    for ln in src:
        if ln.startswith(">"):
            swap = ln.startswith(">A|")
            out.append(ln)
        else:
            out.append(new_seq if swap else ln)
    with open(f"{IN}/{NEW}_{panel}.fasta", "w") as fh:
        fh.write("\n".join(out) + "\n")
    print(f"wrote {NEW}_{panel}.fasta")

wrote pad0_len100_2_fixContacts_0_ternary.fasta
wrote pad0_len100_2_fixContacts_0_alpha.fasta


In [12]:
import os
import pandas as pd

PROJ = "/content/drive/MyDrive/il2_mimetic"
IN   = f"{PROJ}/stage5_complex/inputs"

NEW = "pad0_len100_2_fixContacts_0"
OLD = "pad0_len100_2_fixContacts_3"

seqs = pd.read_csv(f"{PROJ}/stage3_seqs/sequences.tsv", sep="\t")
new_seq = seqs.loc[seqs.seq_id == NEW, "sequence"].iloc[0]
print(f"{NEW}: {len(new_seq)} residues, pos22 = {new_seq[21]}")

for panel in ("ternary", "alpha"):
    src = open(f"{IN}/{OLD}_{panel}.fasta").read().strip().split("\n")
    out, swap = [], False
    for ln in src:
        if ln.startswith(">"):
            swap = ln.startswith(">A|")       # chain A is the design
            out.append(ln)
        else:
            out.append(new_seq if swap else ln)
    dest = f"{IN}/{NEW}_{panel}.fasta"
    with open(dest, "w") as fh:
        fh.write("\n".join(out) + "\n")
    print(f"wrote {os.path.basename(dest)}")

# verify: chain A swapped, receptors identical to the parent
for panel in ("ternary", "alpha"):
    print(f"\n--- {NEW}_{panel}.fasta ---")
    print(open(f"{IN}/{NEW}_{panel}.fasta").read())

pad0_len100_2_fixContacts_0: 115 residues, pos22 = Y
wrote pad0_len100_2_fixContacts_0_ternary.fasta
wrote pad0_len100_2_fixContacts_0_alpha.fasta

--- pad0_len100_2_fixContacts_0_ternary.fasta ---
>A|protein|empty
KLEELKELLLQYEHLLLDLQMYKELIEKLKELLKKVSKEEGEKLKKEFEKKMKDLKSNIKVILDELIKKSSPEEQEILKKMKEDIEKGDTSKLNEYITFMQSKISELKEKIKELE
>B|protein
AVQGTSQFTCFYNSRAQISCVWSQDGALQDTSCQVHAWPDRRRWQQTCELLPVSQASWACNLILGAPDSQKLTTVDIVTLRVLCREGVRWRVMAIQDFKPFENLRLMAPISLQVVHVETHRCNISWEISQASHYFERHLEFEARTLSPGHTWEEAPLLTLKQKQEWICLETLTPDTQYEFQVRVKPLQGEFTTWSPWSQPLAFRTKPAALGKDT
>C|protein
PLPEVQCFVFNVEYMNCTWQSSSEPQPTNLTLHYWYKNSDNDKVQKCSHYLFSEEITSGCQLQKKEIHLYQTFVVQLQDPREPRRQATQMLKLQNLVIPWAPENLTLHKLSESQLELNWNNRFLNHCLEHLVQYRTDWDHSWTEQSVDYRHKFSLPSVDGQKRYTFRVRSRFNPLCGSAQHWSEWSHPIHWGSNTSKEN


--- pad0_len100_2_fixContacts_0_alpha.fasta ---
>A|protein|empty
KLEELKELLLQYEHLLLDLQMYKELIEKLKELLKKVSKEEGEKLKKEFEKKMKDLKSNIKVILDELIKKSSPEEQEILKKMKEDIEKGDTSKLNEYITFMQSKISELKEKIKELE
>B|protein
ELCDDDPPEIPHATFKAMAYKEGTMLNCECKRGFRRIK